In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MP-DE").getOrCreate()

In [0]:
df = spark.read.table('cl_mp_de.`01_bronze`.bronze_crm_customer_base')
display(df)

### Standardizing column names

In [0]:
df_standarized = df.withColumnRenamed('!!Cust_Ref_ID!!','customer_id') \
    .withColumnRenamed('Full_Name','full_name') \
    .withColumnRenamed('[Contact_Info]','contact_information') \
    .withColumnRenamed('Joined_Date_@','joined_date') \
    .withColumnRenamed('Gender_Code','gender')
display(df_standarized)

### Cleaning data

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_extract, initcap, regexp_replace, coalesce, expr, length, row_number
from pyspark.sql.window import Window

#cleaning full_name column
df_clean = df_standarized.withColumn('full_name', regexp_replace('full_name', r"\([^)].*\)", "")) \
        .withColumn('full_name', regexp_replace('full_name', "\\.", "")) \
        .withColumn('full_name', regexp_replace('full_name', "[^a-zA-Z ]", " ")) \
        .withColumn('full_name', initcap(trim(col('full_name'))))

#cleaning joined_date column
df_clean = df_clean.withColumn('joined_date',
        coalesce(
            expr("try_to_date(`joined_date`, 'dd-MM-yyyy')"),
            expr("try_to_date(initcap(trim(`joined_date`)), 'MMM dd, yyyy')"),
            expr("try_to_date(`joined_date`, 'yyyy.MM.dd')"),
        ).cast('date')
)

#cleaning contact_information column
df_clean = df_clean.withColumn('contact_information', when(col('contact_information').isin("NULL"), "Unknown").otherwise(col("contact_information"))) \
    .withColumn('contact_information', regexp_replace('contact_information', "\\+91 ", "")) \
    .withColumn('contact_information', regexp_replace('contact_information', "91-", "")) \
    .withColumn('contact_information', regexp_replace('contact_information', " ", "")) \
    .withColumn('contact_information', when(length(col('contact_information'))>10, "Unknown").otherwise(col("contact_information")))

#removing duplicates
window_spec = Window.partitionBy('customer_id').orderBy(
    when(col('contact_information') == 'Unknown', 2).otherwise(1)
)

df_with_duplicates = df_clean.withColumn("rn", row_number().over(window_spec))
df_filter = df_with_duplicates.filter(col("rn") == 1).drop("rn")
display(df_filter)

### Saving the dataframe as table silver_crm_customer

In [0]:
df_filter.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cl_mp_de.`02_silver`.silver_crm_customer')